# 03. Trajectory Analysis Window Feature Engineering

**Project**: AI-Driven Predictive Geofencing and Trajectory Analysis for Proactive Safety Monitoring  
**Stage**: Chunk 3 — Window-Level Feature Engineering Exploration  

This notebook demonstrates and inspects the spatial, kinematic, geometric, behavioral, and directional entropy features engineered in [`ml/src/features.py`](../src/features.py) across all 676,697 non-overlapping 120-second analysis windows.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Resolve paths dynamically
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'ml' / 'notebooks'
PROCESSED_DIR = NOTEBOOK_DIR.parent / 'data' / 'processed'
FEATURES_PARQUET = PROCESSED_DIR / 'trajectory_features.parquet'
REPORT_FILE = PROCESSED_DIR / 'feature_engineering_report.json'

print(f'Features Parquet: {FEATURES_PARQUET} (exists: {FEATURES_PARQUET.exists()})')
print(f'Quality Report:   {REPORT_FILE} (exists: {REPORT_FILE.exists()})')

## 1. Load Feature Engineering Summary Report

In [ ]:
with open(REPORT_FILE, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(f"Dataset:        {report['dataset_name']}")
print(f"Total Windows:  {report['total_windows']:,}")
print(f"Runtime:        {report['runtime_seconds']:.2f}s")
print('\n=== Quality & Evaluability Flags ===')
for flag, data in report['flags'].items():
    print(f"  {flag:30s}: {data['true_count']:8,d} ({data['true_pct']:6.2f}%)")

## 2. Inspect Sample Window Features & Schema

In [ ]:
df_feat = pq.read_table(FEATURES_PARQUET).slice(0, 100000).to_pandas()
print(f'Loaded sample: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns')
df_feat.head(5)

## 3. Feature Correlation Structure (Kinematics vs. Geometry)

In [ ]:
key_cols = [
    'point_count',
    'path_distance_m',
    'straight_line_displacement_m',
    'tortuosity_index',
    'radius_of_gyration',
    'mean_speed_mps',
    'turn_frequency',
    'heading_variability',
    'loop_tendency',
    'entropy_directional',
]
corr = df_feat[key_cols].corr().round(3)
corr

## 4. Inspection of Directional Entropy (Group E)

Directional entropy measures spatial randomness in compass directions:
- H ~ 0.0: Straight-line, unidirectional movement
- H ~ 1.0: Meandering, directionally uniform wandering

In [ ]:
entropy_series = df_feat['entropy_directional'].dropna()
print('Directional Entropy Summary:')
print(entropy_series.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

## 5. Inspection of Tortuosity & Loop Tendency

Tortuosity and loop tendency distinguish efficient transit from circular wandering.

In [ ]:
print('=== Tortuosity Index ===')
print(df_feat['tortuosity_index'].dropna().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))
print('\n=== Loop / Lapping Tendency ===')
print(df_feat['loop_tendency'].dropna().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))